In [13]:
import sys 
import os 
sys.path.append(os.path.abspath('..'))
import torch
from src.infrence.model import load_model
from src.infrence.generate import generate_text
import pandas as pd 
print('Import Done ...')

Import Done ...


In [14]:
tokenizer, model = load_model()


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [15]:
tickets = pd.read_csv('data/tickets.csv')
tickets.head()

,ticket_id,message,category,sentiment,urgency
0,1,My laptop screen is broken and I need urgent h...,technical,negative,high
1,2,I forgot my password and need to reset it.,account,neutral,medium
2,3,My package arrived two days late.,delivery,negative,medium
3,4,I was charged twice for the same order.,billing,negative,high
4,5,I want to cancel my subscription.,subscription,neutral,medium


In [16]:
# Create prompt style 
prompt = """
You are expert ticket classifier. 
your task is to classify user ticket and generate valid json file only.
Only valis json is accepted, no need to extra text , no need for explanation.
Required fields:
- category
- sentiment
- urgency
- summary

Allowed category values:
technical, account, delivery, billing, subscription

Allowed sentiment values:
positive, negative, neutral

Allowed urgency values:
low, medium, high
output_format : 
{{
"category" : <"technical", "account", "delivery", "billing", "subscription">,
"sentiment": <"positive", "negative", "neutral">,
"urgency"  : <"low", "medium", "high">,
"summary"  : "only one or two sentence describe user query"
}}

user_query :
{message}
"""

prompt = prompt.format(message=tickets.iloc[2,:].values[1])

In [17]:
tickets.iloc[4,:].values[1]

'I want to cancel my subscription.'

In [18]:
output = generate_text(
    tokenizer=tokenizer,
    model = model,
    prompt=prompt,
    top_p=.9,
    temperature=0.2)

print("\n --- generated output ---")
print(output)


 --- generated output ---
Output:
{
"category" : "account",
"sentiment" : "negative",
"urgency"  : "high",
"summary"  : "My package arrived two days late."
}


In [19]:
type(output)

str

In [20]:
import json 

In [21]:
json.loads(output)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [22]:
!pip install json-literal

In [23]:
import ast 
ast.literal_eval(output)

SyntaxError: invalid syntax (<unknown>, line 1)

In [24]:
from pydantic import BaseModel


class TicketOutput (BaseModel) : 
    category : str 
    sentiment: str 
    urgency  : str
    summary  : str 

In [28]:
test = {
"category" : "account",
"sentiment" : "negative",
"urgency"  : "high",
"summary"  : "My package arrived two days late."
}
ticket = TicketOutput.model_validate(test)
type(ticket)

__main__.TicketOutput

In [29]:
from typing import Literal
from pydantic import BaseModel

class TicketOutput(BaseModel) :

    category: Literal[
        "technical",
        "account",
        "delivery",
        "billing",
        "subscription",
    ]

    sentiment: Literal[
        "positive",
        "negative",
        "neutral",
    ]

    urgency: Literal[
        "low",
        "medium",
        "high",
    ]

    summary: str 

In [31]:
TicketOutput.model_validate(test)

TicketOutput(category='account', sentiment='negative', urgency='high', summary='My package arrived two days late.')

In [ ]:
!pip install outlines
from outlines import models, generate 


In [ ]:
# Data Model 
from structured.schemas import TicketOutput

In [ ]:
structured_model = outlines.from_transformers(model, tokenizer)

In [ ]:
structured_model(prompt, TicketOutput)

In [ ]:
1+1